# Near-Duplicate Clustering Evaluation

This notebook evaluates the near-duplicate detection pipeline using **DBSCAN** on
image embeddings. It reports precision, recall, and F1 against ground-truth
duplicate groups, and visualises the distribution of pairwise similarity scores.

## Dataset setup — INRIA Holidays

1. Download the **INRIA Holidays** dataset from
   <http://lear.inrialpes.fr/~jegou/data.php>.
2. Extract and place images under:

```
photo-declutterer/
└── data/
    └── holidays/
        ├── images/          ← all holiday JPEG images (1491 images)
        └── holidays.dat     ← ground-truth group file (from the official release)
```

3. If you only want a quick smoke-test, set `SAMPLE_DIR` to any small folder of
   photos that contains obvious duplicates — the pipeline will compute embeddings
   on the fly and skip the parquet cache.

> **Reference:** Jégou, H., Douze, M. & Schmid, C. (2008).
> *Hamming Embedding and Weak Geometric Consistency for Large Scale Image Search.*
> ECCV 2008.

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("dedup_eval")

HOLIDAYS_DIR  = PROJECT_ROOT / "data" / "holidays" / "images"
GT_FILE       = PROJECT_ROOT / "data" / "holidays" / "holidays.dat"
RESULTS_DIR   = PROJECT_ROOT / "results"
CACHE_PARQUET = RESULTS_DIR / "dedup_embeddings.parquet"

# Fallback: use a small local sample folder for quick tests
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
log.info("Paths configured. PROJECT_ROOT=%s", PROJECT_ROOT)

In [ ]:
from src.dedup_cluster import compute_embeddings

if CACHE_PARQUET.exists():
    log.info("Loading cached embeddings from %s", CACHE_PARQUET)
    results_df = pd.read_parquet(CACHE_PARQUET)
else:
    img_dir = HOLIDAYS_DIR if HOLIDAYS_DIR.exists() else SAMPLE_DIR
    log.info("Computing embeddings for images in %s …", img_dir)
    results_df = compute_embeddings(
        image_dir=str(img_dir),
        model_name="efficientnet_b0",   # backbone used for embedding extraction
        batch_size=64,
        img_size=(224, 224),
    )
    results_df.to_parquet(CACHE_PARQUET, index=False)
    log.info("Embeddings saved to %s (%d images)", CACHE_PARQUET, len(results_df))

print(results_df.head())
print(f"\nShape: {results_df.shape}")

In [ ]:
from src.dedup_cluster import run_dedup_clustering, evaluate_clustering

# ── Run DBSCAN near-duplicate clustering ─────────────────────────────────────
log.info("Running DBSCAN dedup clustering …")
cluster_df = run_dedup_clustering(
    embeddings_df=results_df,
    eps=0.15,           # cosine-distance threshold for duplicate neighbourhood
    min_samples=2,      # at least 2 images to form a duplicate group
    metric="cosine",
)

n_clusters  = cluster_df["Cluster"].nunique()
n_noise     = (cluster_df["Cluster"] == -1).sum()
log.info("DBSCAN found %d duplicate groups (+%d singletons/noise)", n_clusters, n_noise)
print(cluster_df[["Filename", "Cluster", "Similarity_Score"]].head(20))

In [ ]:
# ── Evaluate against ground-truth (requires holidays.dat) ────────────────────
if GT_FILE.exists():
    metrics = evaluate_clustering(
        cluster_df=cluster_df,
        ground_truth_file=str(GT_FILE),
    )
    print("\nClustering metrics:")
    for k, v in metrics.items():
        print(f"  {k:20s}: {v:.4f}")
else:
    log.warning(
        "Ground-truth file not found (%s). Skipping metric computation.", GT_FILE
    )
    print("[INFO] Ground-truth file missing — metrics skipped.")

# ── Histogram of Similarity_Score values ─────────────────────────────────────
scores = cluster_df["Similarity_Score"].dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores, bins=50, color="steelblue", edgecolor="white", linewidth=0.5)
ax.axvline(x=1 - 0.15, color="crimson", linestyle="--", label="eps threshold (0.85)")
ax.set_xlabel("Similarity Score (cosine)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Pairwise Similarity Scores")
ax.legend()
plt.tight_layout()
fig_path = RESULTS_DIR / "dedup_similarity_histogram.png"
plt.savefig(str(fig_path), dpi=150)
plt.show()
log.info("Histogram saved to %s", fig_path)